In [ ]:
!pip install langchain langchain-community langchain-groq faiss-cpu sentence-transformers PyPDF2 pypdf --q

In [ ]:
!pip install fpdf --q

In [ ]:
from fpdf import FPDF
import os

pdf=FPDF()
pdf.add_page()
pdf.set_font("Arial",size=12)

context=["""
   "THE ECHO IN THE WALLS"

"A Short Horror Story"

"The grandfather clock in the downstairs hallway struck three in the morning, its heavy chime vibrating through the floorboards of Blackwood Manor."

"For Arthur, the sound was an unwelcome reminder of his growing insomnia."

"He had purchased the remote, centuries-old estate three months prior, seeking isolation to complete his historical research."

"Instead, he had found an oppressive, suffocating silence—except for the scratching."

"- It had started a week ago as a faint, rhythmic scraping behind the plaster of his bedroom wall."

"At first, he dismissed it as mice nesting before the winter frost."

"He placed traps and scattered poison, but the noises only grew louder, transitioning from the frantic skittering of rodents to something slower, more deliberate."

"It sounded like fingernails dragging across raw wood."

"Tonight, the sound was closer than ever, originating from the small, locked door tucked beneath the main staircase."

"The door had no key; its iron handle was rusted solid, fused into its plate by decades of neglect."

"Arthur stood before it now, holding a flickering candle that cast long, dancing shadows against the cracked wallpaper."

"He pressed his ear against the cold timber."

"From the other side came a soft, rhythmic breathing."

"It was heavy, wet, and unmistakably human."

"‘Is someone there?’ Arthur whispered, cursing himself for the tremor in his voice."

"He received no verbal reply, but the scratching stopped instantly."

"A long, agonizing silence followed, filled only by the thumping of his own heart."

"Then, three sharp knocks rattled the door from within."

"Thump."

"Terror, cold and absolute, gripped his chest."

"Blackwood Manor was supposed to have been abandoned for forty years after the mysterious disappearance of the Blackwood family."

"The local archives mentioned a hidden cellar beneath the foundations, a space sealed off during the plague of the late nineteenth century and never reopened."

"Arthur retreated to his study, locking the heavy oak door behind him, but he did not sleep."

"By daylight, the terror faded into a grim obsession."

"Armed with a crowbar and a heavy iron hammer from the shed, Arthur stood before the sealed door once more."

"The midday sun streamed through the dusty windows, giving him a false sense of security."

"He slammed the crowbar into the seam of the door frame."

"The wood groaned and splintered, releasing a blast of stale, stagnant air that smelled of damp earth and ancient rot."

"With a final, violent heave, the lock gave way, and the door swung open into pitch blackness."

"A steep flight of stone steps descended into the earth, disappearing into an absolute dark that the daylight refused to penetrate."

"Holding a high-powered flashlight, Arthur began his descent."

"The air grew progressively colder with every step, turning his breath into faint white plumes."

"The walls changed from brick to rough-hewn stone, weeping with moisture."

"At the bottom of the stairs lay a vast, vaulted chamber."

"It was empty, save for a massive, iron-bound mirror standing in the center of the room."

"The mirror was pristine, untouched by dust or grime, reflecting the beam of his flashlight with silver brilliance."

"Arthur approached it cautiously, his boots crunching on the damp floor."

"He raised the light to face the glass, expecting to see his own disheveled reflection."

"He saw the room."

"He saw the stone walls, the stairs behind him, and the beam of his flashlight cutting through the dark."

"But the reflection of Arthur himself was missing."

"The space where he stood in front of the mirror was entirely empty in the glass."

"Panic surged through him, and he stumbled backward."

"As he did, a shape began to form within the mirror’s reflection."

"Out of the darkness behind where his reflection should have been, a figure emerged."

"It walked with a strange, jointed limp, its skin a sickening, translucent pale, and its eyes entirely hollowed out, leaving dark voids."

"The entity in the mirror smiled—a wide, unnatural grin that stretched too far across its face, revealing rows of needle-thin teeth."

"It raised a hand, pressing its pale, elongated fingers against the glass from the inside."

"Simultaneously, Arthur felt a freezing pressure against the back of his neck."

"The scratching sound returned, loud and deafening, echoing not from the walls, but from inside his own skull."

"He turned around frantically, shining his light into the darkness of the cellar, but there was nothing there."

"When he looked back at the mirror, the entity was gone."

"Instead, Arthur saw himself standing in the mirror."

"His reflection was staring back at him, screaming in silent agony, throwing its hands against the glass, begging to be let out."

"Arthur touched his own face, but his skin felt cold, stiff, and wet."

"He tried to speak, but no sound escaped his throat."

"Slowly, the perspective shifted."

"Arthur realized he was no longer looking at a mirror."

"He was looking out through a window."

"He was inside the glass, trapped in the freezing, silent dark of the reflection, while the pale, wide-grinning thing walked backward up the stone stairs, occupying his body, carrying his flashlight, and closing the heavy cellar door behind it, locking him in the dark forever."

"""
]
context[0] = context[0].replace("•", "-")
context[0] = context[0].replace("–", "-")
context[0] = context[0].replace("—", "-")  # Add this line
context[0] = context[0].replace("’", "'")
context[0] = context[0].replace("‘", "'")
context[0] = context[0].replace("“", '"')
context[0] = context[0].replace("”", '"')
for line in context:
  pdf.multi_cell(0,10,line)
pdf_file="Story.pdf"
pdf.output(pdf_file)
print(f"PDF saved as {pdf_file}")

os.listdir(".")

PDF saved as Story.pdf


['.config', 'Story.pdf', 'sample_data']

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import  CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import  FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]="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"

loader=PyPDFLoader("Story.pdf")
pages=loader.load()
splitter=CharacterTextSplitter(chunk_size=500,chunk_overlap=50)
document=splitter.split_documents(pages)
embedding=HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db=FAISS.from_documents(document,embedding)

llm=ChatGroq(model="llama-3.1-8b-instant")
prompt=PromptTemplate.from_template("Use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {question}")

parser=StrOutputParser()

def ask_pdf(query: str):
  docs=db.similarity_search_with_score(query,k=3)
  context="\n\n".join([d[0].page_content for d in docs])
  chain=prompt|llm|parser
  return chain.invoke({"context": context, "question": query})

while True:
  query=input("\nAsk a question (or type 'exit'): ")
  if query.lower()=="exit":
    break
  answer=ask_pdf(query)
  print("\nAnswer:\n",answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Ask a question (or type 'exit'): who is the hero of the story

Answer:
 The hero of the story is Arthur. He is the protagonist who experiences the terrifying events in the story. He is the one who is trapped in the mirror, tries to escape, and interacts with the supernatural entity and his own reflection. Throughout the story, Arthur's actions and reactions drive the plot and his emotions and thoughts are explored.

Ask a question (or type 'exit'): one line of the story

Answer:
 "He saw the room.

Ask a question (or type 'exit'): twist in the story

Answer:
 The twist in the story is that Arthur is not trapped outside the mirror, but rather, he is trapped inside the mirror, and the entity and the reflection he has been seeing are actually his own alternate selves or personas trying to break free from the mirror.

As Arthur approaches the mirror, he expects to see his own reflection, but instead, he sees the room and himself missing from the glass. This suggests that Arthur is the one

In [ ]:
!pip install gradio

In [1]:
import gradio as gr

def calculate(a, b, op):
    if op == "+":
        return a + b
    elif op == "-":
        return a - b
    elif op == "*":
        return a * b
    elif op == "/":
        return a / b if b != 0 else "Cannot divide by zero"

gr.Interface(
    fn=calculate,
    inputs=[
        gr.Number(),
        gr.Number(),
        gr.Radio(["+", "-", "*", "/"])
    ],
    outputs="text"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa861f1b945ca60079.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
